SDXL-BERT HYBRID PIPELINE FOR TEXT TO IMAGE


In [ ]:
# -*- coding: utf-8 -*-
# @title Setup: Install Dependencies and Download Models (Run First!)
# Install required libraries
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install diffusers transformers accelerate safetensors spacy Pillow numpy tqdm ftfy clip-interrogator -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 91.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 44.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 119.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 13.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.7/848.7 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# @title Download spaCy model (en_core_web_lg)
# Check if the model is already installed to avoid redundant downloads
import spacy.cli
import sys
import subprocess
import logging

# Setup basic logging for installation steps
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

SPACY_MODEL_NAME = "en_core_web_lg"

try:
    spacy.load(SPACY_MODEL_NAME)
    logging.info(f"SpaCy model '{SPACY_MODEL_NAME}' already installed.")
except OSError:
    logging.warning(f"SpaCy model '{SPACY_MODEL_NAME}' not found. Downloading...")
    try:
        # Using subprocess ensures it runs correctly in Colab/notebook environments
        subprocess.check_call([sys.executable, "-m", "spacy", "download", SPACY_MODEL_NAME])
        logging.info(f"Successfully downloaded {SPACY_MODEL_NAME}.")
        # Verify load after download
        spacy.load(SPACY_MODEL_NAME)
        logging.info(f"Successfully loaded {SPACY_MODEL_NAME} after download.")
    except Exception as e:
        logging.error(f"Failed to download or load spaCy model '{SPACY_MODEL_NAME}': {e}", exc_info=True)
        # You might want to raise an error or handle this case depending on script requirements
        raise RuntimeError(f"Could not load spaCy model '{SPACY_MODEL_NAME}'.")



In [ ]:
!pip uninstall torch torchvision torchaudio fastai -y


Found existing installation: torch 2.6.0+cu118
Uninstalling torch-2.6.0+cu118:
  Successfully uninstalled torch-2.6.0+cu118
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Found existing installation: fastai 2.7.19
Uninstalling fastai-2.7.19:
  Successfully uninstalled fastai-2.7.19


In [ ]:
!pip install torch==2.6.0+cu118 torchvision==0.21.0+cu118 torchaudio==2.6.0+cu118 --index-url https://download.pytorch.org/whl/cu118


Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download.pytorch.org/whl/cu118/torch-2.6.0%2Bcu118-cp311-cp311-linux_x86_64.whl.metadata (27 kB)
Using cached https://download.pytorch.org/whl/cu118/torch-2.6.0%2Bcu118-cp311-cp311-linux_x86_64.whl (848.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 5.1 MB/s eta 0:00:00


In [1]:
#@title Main pipeline code (fallback mechanisms set in place)
!export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
!python sdxl_improved.py "A modern cozy bedroom with a Scandinavian design aesthetic, featuring a warm, inviting atmosphere. The main wall is painted a soft teal blue, with one side wall in light gray and a darker gray accent wall.A red fabric sofa bed is placed in the center, made up neatly with white bedding and a cream-colored throw blanket.Behind the bed, there is a tall white open bookshelf filled with books (mainly in red, orange, and neutral colors), small decor items, and a potted plant at the top.A sleek white floor lamp with a minimalistic conical shade stands next to the bookshelf.A black modern clock with white hands hangs on the teal wall, along with a cluster of framed photographs and minimalist art pieces arranged in a gallery style.To the right, a wicker chair with a curved back and metal legs is placed near the window, topped with a striped cushion in earthy tones (red, brown, beige).In front of the chair, there is a small footstool with a turquoise cushion and black thin metal legs.A square wooden coffee table with a lattice design sits at the center, holding a small potted plant, a couple of books, and light decor items.A woven round side table with a wooden top is placed beside the bed, holding a water bottle and books.Full-height white curtains cover large windows, letting in abundant natural sunlight, casting soft shadows.The flooring is dark wood with a beige rug underneath the bed and table area.The overall mood is bright, fresh, minimalistic, and cozy." --seed 12345

python3: can't open file '/content/sdxl_improved.py': [Errno 2] No such file or directory
